# Chapter 6. 강화학습 알고리즘 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter06_q_learning.ipynb)

책 본문: [Chapter 6](https://smhanlab.com/book-ml/kor/ml2/chapter06.html)

Q-learning이 환경의 전이확률을 전혀 모른 채로도, 직접 행동해보고
관찰한 결과만으로 최적 정책을 학습하는 것을 실제로 확인합니다.

## 1. 장난감 환경: 2칸짜리 복도

State 0, State 1이 있고, 각 상태에서 두 행동이 가능합니다:
- `stay`(0): 제자리, 보상 -1
- `forward`(1): 한 칸 전진, 보상 -1 (단, State 1에서 forward하면 **종료**, 보상 +10)

최적 정책은 당연히 "항상 forward"입니다 — Q-learning이 탐험만으로
이걸 스스로 알아내는지 봅니다.

In [ ]:
def transition(s, a):
    # 반환: (reward, next_state).  next_state == -1 이면 에피소드 종료.
    if a == 1:  # forward
        if s == 1:
            return 10, -1  # 목표 도달, 종료
        return -1, s + 1
    return -1, s  # stay

## 2. epsilon-greedy + Q-learning 업데이트 (책 6.6~6.7절 코드)

In [ ]:
import random

def epsilon_greedy(Q, s, epsilon, n_actions):
    if random.random() < epsilon:
        return random.randrange(n_actions)
    return max(range(n_actions), key=lambda a: Q[s][a])

def q_learning_train(transition, n_states, n_actions, n_episodes, alpha, gamma, epsilon):
    Q = [[0.0] * n_actions for _ in range(n_states)]
    for episode in range(n_episodes):
        s = 0
        for _ in range(20):
            a = epsilon_greedy(Q, s, epsilon, n_actions)
            r, s_next = transition(s, a)
            max_next_q = max(Q[s_next]) if s_next != -1 else 0.0
            td_error = r + gamma * max_next_q - Q[s][a]
            Q[s][a] += alpha * td_error
            if s_next == -1:
                break
            s = s_next
    return Q

## 3. 학습 실행 + 결과 확인

In [ ]:
random.seed(0)
Q = q_learning_train(transition, n_states=2, n_actions=2,
                      n_episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1)

for s in range(2):
    print(f"Q({s}, stay)    = {Q[s][0]:.3f}")
    print(f"Q({s}, forward) = {Q[s][1]:.3f}")

greedy_policy = [max(range(2), key=lambda a: Q[s][a]) for s in range(2)]
print("\n학습된 그리디 정책:", ["stay" if a == 0 else "forward" for a in greedy_policy])
assert greedy_policy == [1, 1], "두 상태 모두 forward를 골라야 합니다"
print("두 상태 모두 forward(전진)를 선택 — 최적 정책을 스스로 찾았습니다.")